# Physical validation -- DIII-D, $N_x=0$, 151x261 mesh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BecerraMiguel/SemiFree-Solver/blob/main/notebooks/01_physical_validation.ipynb)

The coil currents of the 18 DIII-D PF coils are computed without X-points, and the solution is checked for
(i) the boundary condition $\psi_b=\psi_{ref}=0$, (ii) coil currents of a physically reasonable size, and
(iii) a consistent magnetic field. The poloidal field is evaluated in two independent ways -- with the closed-form
Green functions $G^{B_R},G^{B_Z}$, and by differentiating $\psi$
($B_R=-\frac{1}{R}\partial_Z\psi$, $B_Z=\frac{1}{R}\partial_R\psi$) -- and their divergences
$\nabla\cdot\mathbf{B}$ are compared.

**Outputs:** boundary residual, coil currents, $\psi(R,Z)$, $\mathbf{B}$ (Green functions and derivative of
$\psi$), their difference and $\nabla\cdot\mathbf{B}$ for each.

**Estimated time (Colab, 2 cores):** ~1.5 min build + ~8 min solver (the 151x261 run includes the dense
$B_R$ and $B_Z$ sweeps) = **~10 minutes**.


**DIII-D case** (same for every mesh): $R_0=1.67$ m, $a=0.67$ m, $\kappa=1.77$, $\delta=0.30$,
$I_p=1.5$ MA, $P_{axis}=50$ kPa, $B_{axis}=2.0$ T, $\Psi_b=0$, with $N_c=18$ PF coils.
Computational domain: $R\in[0.15,\,3.0]$ m, $Z\in[-1.75,\,1.75]$ m.

The inputs ($J_\phi$, boundary, coil positions) are shipped with the repository in `cases/DIII-D_<mesh>/`, and the
solver configuration in `configs/DIII-D_<mesh>.json`. This notebook clones the repository, builds the solver and
runs everything from there: nothing has to be uploaded.

This notebook uses the **151x261** mesh, with $J_\phi$ from the fixed-boundary Grad-Shafranov solver
(`cases/DIII-D_151x261/Jt.txt`). Reference values are printed next to the computed ones.


In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BecerraMiguel/SemiFree-Solver.git'
REPO_REF = None        # branch or tag to clone (None = default branch)
REPO_DIR = os.environ.get('REPRODUCE_REPO_DIR', '/content/SemiFree-Solver')
if not os.path.isdir(REPO_DIR):
    cmd = ['git', 'clone', '--depth', '1'] + (['--branch', REPO_REF] if REPO_REF else [])
    subprocess.run(cmd + [REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f'{REPO_DIR}/notebooks')
import reproduce_common as rc
import numpy as np
import matplotlib.pyplot as plt

USE_GOOGLE_DRIVE = False   # True: keep results in Google Drive (survive disconnections, shared between notebooks)

WORK = rc.default_work_dir(USE_GOOGLE_DRIVE)
os.makedirs(f'{WORK}/figures', exist_ok=True)

rc.environment_report()
print('Working directory:', WORK)

## 1. Build
The solver is built with `make` (the repository `Makefile` enables `-fopenmp`). This notebook needs the full version, which also writes `BR_check.txt` and `BZ_check.txt`.

In [ ]:
BIN = rc.build_semifree(REPO_DIR, WORK, skip_bfield=False)

## 2. Run the semi-free solver ($N_x=0$, 151x261)

In [ ]:
TAG = '151x261'
SUB = 'Nx0_full_field'
rc.run_semifree(BIN, REPO_DIR, WORK, 'Nx0', TAG, subdir=SUB, need_bfield=True)
run = rc.load_run(REPO_DIR, WORK, 'Nx0', TAG, subdir=SUB)
assert run is not None and run['BRg'] is not None, 'solver outputs are missing'
print('Outputs in:', run['dir'])

## 3. Boundary condition and coil currents

In [ ]:
rc.physical_summary(run)
print()
print(f'{"Coil":>5} {"R [m]":>8} {"Z [m]":>8} {"I [kA]":>10}')
for i, (r, z, cur) in enumerate(run['currents']):
    print(f'{i:>5} {r:>8.3f} {z:>8.3f} {cur/1e3:>10.1f}')

## 4. Poloidal flux $\psi(R,Z)$

In [ ]:
F = rc.field_arrays(run)
rc.plot_flux(F, run, fname=f'{WORK}/figures/physical_flux.png')
plt.show()

## 5. Magnetic field from two methods
Green functions (first) and derivative of $\psi$ (second). At this scale they are visually indistinguishable.

In [ ]:
rc.plot_field(F, run, F['BRg'], F['BZg'], r'DIII-D: $\mathbf{B}$ (Green functions)',
              fname=f'{WORK}/figures/physical_B_green.png')
plt.show()
rc.plot_field(F, run, F['BRf'], F['BZf'], r'DIII-D: $\mathbf{B}$ (derivative of $\psi$)',
              fname=f'{WORK}/figures/physical_B_derivative.png')
plt.show()

## 6. Difference between the two methods
The discrepancy concentrates in a narrow band just inside the plasma boundary, where $G^{B_R}$ and $G^{B_Z}$ are more singular than $G^\psi$.

In [ ]:
diff, fig = rc.plot_field_difference(F, run, fname=f'{WORK}/figures/physical_diff_B.png')
plt.show()
inside, dcoil, h = rc.field_statistics(F, run, diff)

## 7. Divergence of $\mathbf{B}$
The field obtained by differentiating $\psi$ satisfies $\nabla\cdot\mathbf{B}=0$ to machine precision; the Green-function field loses accuracy in the same inner band (and next to the coils, because of their singularity).

In [ ]:
div_g = rc.divergence(F, F['BRg'], F['BZg'])
div_f = rc.divergence(F, F['BRf'], F['BZf'])
far = dcoil > 3 * h
print(f'max |div B|, derivative of psi      : {np.nanmax(np.abs(div_f)):.2e}   (reference: ~1e-14 over the whole domain)')
print(f'max |div B|, Green (whole domain)   : {np.nanmax(np.abs(div_g)):.2e}')
print(f'mean |div B|, Green, plasma interior: {np.nanmean(np.abs(div_g[inside])):.2e}')
print(f'mean |div B|, Green, outside plasma and away from coils: {np.nanmean(np.abs(div_g[(~inside) & far])):.2e}')
rc.plot_divergence(F, run, div_g, div_f, far,
                   fname_raw=f'{WORK}/figures/physical_divB_unfiltered.png',
                   fname_excl=f'{WORK}/figures/physical_divB_no_coils.png')
plt.show()

## 8. Results bundle (optional)
Packages outputs and figures in a `.zip` and, on Colab, starts its download.

In [ ]:
import glob
zp = rc.zip_results(WORK, ['Nx0_full_field'], f'{WORK}/results_physical_validation.zip',
                    extra_files=sorted(glob.glob(f'{WORK}/figures/physical_*.png')))
rc.offer_download(zp)